# E-Commerce Customer Return Behavior — Data Cleaning

**Capstone Project | Data Quality & Cleaning Notebook**

This notebook documents the full data cleaning process for the customer return behavior dataset (`train.csv`, 200,000 orders, 16 fields).

Workflow:
1. Load and inspect the raw data
2. Check for structural issues (missing values, duplicates)
3. Check for logically impossible values (negatives, out-of-range values)
4. Decide, with reasoning, how to treat each issue
5. Apply the cleaning and verify the result
6. Save the analysis-ready dataset

Each cleaning decision is explained in a markdown cell immediately above the code that implements it, so the logic is traceable to the [Data Quality Report](.) delivered alongside this notebook.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df = pd.read_csv("/Users/venugopalnaidupotturi/Downloads/train.csv")
print(df.shape)
df.head()


(200000, 16)


,customer_age,product_price,discount_percent,product_rating,past_purchase_count,past_return_rate,delivery_delay_days,session_length_minutes,num_product_views,device_type,product_category,shipping_method,payment_method,used_coupon,returned,order_id
0,25,11.666137,43.559601,1.933533,8,0.331408,-0.089637,3.273029,26,tablet,sports,express,paypal,1,0,75381
1,76,31.378967,66.526039,0.690237,9,0.105203,3.639277,130.513123,8,mobile,electronics,standard,credit_card,1,0,65569
2,40,25.460603,20.762374,2.347620,14,0.236683,0.294060,92.780917,-2,desktop,toys,express,credit_card,0,0,163473
3,62,13.517170,25.669260,2.033596,9,0.410038,1.683333,103.004177,7,mobile,clothing,express,apple_pay,0,0,90518
4,51,75.771311,15.527927,4.907556,9,0.123565,-0.153691,100.752658,6,mobile,electronics,standard,credit_card,1,0,138866


## 1. Structural Checks

Before touching any values, confirm the dataset is structurally sound: correct shape, expected data types, no missing values, and no duplicate records. These are "hygiene" checks that should always run first — if they fail, everything downstream is unreliable.


In [2]:
print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)


Shape: (200000, 16)

Data types:
customer_age                int64
product_price             float64
discount_percent          float64
product_rating            float64
past_purchase_count         int64
past_return_rate          float64
delivery_delay_days       float64
session_length_minutes    float64
num_product_views           int64
device_type                   str
product_category              str
shipping_method               str
payment_method                str
used_coupon                 int64
returned                    int64
order_id                    int64
dtype: object


In [3]:
# Missing values — none expected, but always verify rather than assume
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Total missing cells:", df.isnull().sum().sum())


Missing values per column:
customer_age              0
product_price             0
discount_percent          0
product_rating            0
past_purchase_count       0
past_return_rate          0
delivery_delay_days       0
session_length_minutes    0
num_product_views         0
device_type               0
product_category          0
shipping_method           0
payment_method            0
used_coupon               0
returned                  0
order_id                  0
dtype: int64

Total missing cells: 0


In [4]:
# Duplicate records — check both full-row duplicates and duplicate primary keys (order_id)
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate order_id values:", df['order_id'].duplicated().sum())


Fully duplicated rows: 0
Duplicate order_id values: 0


**Result:** 200,000 rows, 16 columns, zero missing values, zero duplicate rows, zero duplicate order IDs. The dataset is structurally complete — the remaining work is about value-level plausibility, not missing/duplicate data.

## 2. Categorical Consistency Check

Before checking numeric plausibility, confirm the four categorical fields don't have inconsistent labels (different casing, stray whitespace, near-duplicate categories like `"Mobile"` vs `"mobile"`) that would silently fragment groups during analysis.


In [5]:
cat_cols = ['device_type', 'product_category', 'shipping_method', 'payment_method']
for c in cat_cols:
    print(f"--- {c} ({df[c].nunique()} unique) ---")
    print(df[c].value_counts(dropna=False))
    print()


--- device_type (3 unique) ---
device_type
mobile     109371
desktop     48903
tablet      41726
Name: count, dtype: int64

--- product_category (6 unique) ---
product_category
toys           48287
beauty         37100
electronics    36092
home           31839
clothing       23913
sports         22769
Name: count, dtype: int64

--- shipping_method (3 unique) ---
shipping_method
standard    131311
express      53088
same_day     15601
Name: count, dtype: int64

--- payment_method (4 unique) ---
payment_method
debit_card     56667
credit_card    54743
apple_pay      47518
paypal         41072
Name: count, dtype: int64



**Result:** all four categorical fields are already clean — lowercase, no whitespace issues, no near-duplicate labels. We'll still apply a defensive `.str.strip().str.lower()` during cleaning in case this notebook is ever re-run on a fresh export that isn't as clean.

## 3. Numeric Plausibility Check

This is the core of the data quality investigation. For each numeric field, the question is: **given what this field represents in the real world, what values are actually possible?**

For example, a product price cannot be negative, a star rating is bounded to a fixed scale, and a percentage cannot exceed reasonable bounds. `describe()` surfaces the min/max needed to test this.


In [6]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
customer_age,200000.0,56.479545,17.952930,10.000000,44.000000,58.000000,72.000000,87.000000
product_price,200000.0,37.984234,45.626883,-183.292092,6.876686,24.461142,53.355820,855.078645
discount_percent,200000.0,42.872056,21.510490,-7.540832,25.848785,42.945197,62.642248,78.791547
product_rating,200000.0,2.812805,1.219874,0.445342,1.789263,2.818731,3.810169,5.452690
past_purchase_count,200000.0,9.620210,3.139979,1.000000,8.000000,9.000000,10.000000,26.000000
past_return_rate,200000.0,0.215114,0.137162,-0.060878,0.118326,0.191392,0.282027,0.866420
delivery_delay_days,200000.0,0.625050,2.331221,-9.049981,-0.751862,0.519040,1.896547,7.961333
session_length_minutes,200000.0,77.406138,36.394216,-11.010479,48.995295,81.562252,108.209400,137.652911
num_product_views,200000.0,14.654450,12.659400,-5.000000,3.000000,13.000000,25.000000,44.000000
used_coupon,200000.0,0.615400,0.486502,0.000000,0.000000,1.000000,1.000000,1.000000


Several fields show **impossible minimums**:

| Field | Min observed | Why it's impossible |
|---|---|---|
| `product_price` | -183.29 | A price cannot be negative |
| `discount_percent` | -7.54 | A discount cannot be negative |
| `product_rating` | (max 5.45) | Rating scale is 0-5; values above 5 are impossible |
| `past_return_rate` | -0.06 | A rate/proportion cannot be negative |
| `session_length_minutes` | -11.01 | A duration cannot be negative |
| `num_product_views` | -5 | A count cannot be negative |

`delivery_delay_days` (min -9.05) and `customer_age` (min 10) look unusual too, but are handled differently below — a negative delay and a young age are both *plausible*, not impossible, so they need a judgment call rather than a mechanical fix.

Let's quantify exactly how many rows are affected in each impossible-value field.


In [7]:
impossible_cols = ['product_price', 'discount_percent', 'past_return_rate', 'num_product_views']

for c in impossible_cols:
    n_bad = (df[c] < 0).sum()
    print(f"{c:25s} negative rows: {n_bad:6d}  ({n_bad/len(df)*100:5.2f}%)")

n_high_rating = (df['product_rating'] > 5).sum()
print(f"{'product_rating':25s} rows > 5:      {n_high_rating:6d}  ({n_high_rating/len(df)*100:5.2f}%)")


product_price             negative rows:  27285  (13.64%)
discount_percent          negative rows:   1686  ( 0.84%)
past_return_rate          negative rows:   2220  ( 1.11%)
num_product_views         negative rows:  21443  (10.72%)
product_rating            rows > 5:        3831  ( 1.92%)


## 4. Is the Noise Meaningful, or Random?

Before blindly correcting these values, it's worth checking whether the negative values carry a real signal (e.g. "negative price" might secretly mean "refunded item" and correlate strongly with returns) or whether they're just random noise layered on top of true values.

The test: compare the return rate *among the affected rows* to the overall return rate. If they're close, the negative values aren't informative and it's safe to correct them.


In [21]:
overall_rate = df['returned'].mean()
print(f"Overall return rate: {overall_rate:.3f}\n")

for c in ['product_price', 'num_product_views', 'session_length_minutes',
          'discount_percent', 'past_return_rate']:
    flagged_rate = df.loc[df[c] < 0, 'returned'].mean()
    print(f"{c:25s} return rate when negative: {flagged_rate:.3f}  (overall: {overall_rate:.3f})")


Overall return rate: 0.475

product_price             return rate when negative: 0.438  (overall: 0.475)
num_product_views         return rate when negative: 0.507  (overall: 0.475)
session_length_minutes    return rate when negative: 0.515  (overall: 0.475)
discount_percent          return rate when negative: 0.415  (overall: 0.475)
past_return_rate          return rate when negative: 0.422  (overall: 0.475)


**Result:** the return rate among rows with impossible negative values (41-52%) sits close to the overall rate (47.5%) for every field tested. This confirms the negative values don't carry distinct information — they behave like random measurement noise, not a meaningful "refund flag" or similar. It's safe to correct them without losing signal.

## 5. Outlier Scan (Separate From Errors)

Impossible values (Section 3) are different from **statistical outliers** — a $600 order or a customer with 20 past purchases is unusual but not impossible. We use the IQR method to quantify extremes across all numeric fields, purely to decide whether any of them need special handling later (e.g. capping for chart readability), not to delete them.


In [22]:
num_cols = ['customer_age', 'product_price', 'discount_percent', 'product_rating',
            'past_purchase_count', 'past_return_rate', 'delivery_delay_days',
            'session_length_minutes', 'num_product_views']

for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f"{c:25s} IQR outliers: {out:6d} ({out/len(df)*100:5.2f}%)  bounds=({lo:.2f}, {hi:.2f})")


customer_age              IQR outliers:      0 ( 0.00%)  bounds=(2.00, 114.00)
product_price             IQR outliers:  11689 ( 5.84%)  bounds=(-62.84, 123.07)
discount_percent          IQR outliers:      0 ( 0.00%)  bounds=(-29.34, 117.83)
product_rating            IQR outliers:      0 ( 0.00%)  bounds=(-1.24, 6.84)
past_purchase_count       IQR outliers:  23432 (11.72%)  bounds=(5.00, 13.00)
past_return_rate          IQR outliers:   7260 ( 3.63%)  bounds=(-0.13, 0.53)
delivery_delay_days       IQR outliers:   9053 ( 4.53%)  bounds=(-4.72, 5.87)
session_length_minutes    IQR outliers:      0 ( 0.00%)  bounds=(-39.83, 197.03)
num_product_views         IQR outliers:      0 ( 0.00%)  bounds=(-30.00, 58.00)


**Decision:** these outliers (e.g. 11.7% of `past_purchase_count`, 5.8% of `product_price`) are retained as-is. They represent plausible business extremes (loyal repeat customers, premium products), not data errors, so removing them would throw away real variation the model/analysis needs. They may be capped for *display only* later during visualization, but not removed from the underlying data.

## 6. Cleaning Decision

Summary of what will be corrected and what will be deliberately left alone:

| Field | Action | Why |
|---|---|---|
| `product_price`, `discount_percent`, `past_return_rate`,session_length_minutes, `num_product_views` | **Floor negative values to 0** | Impossible values, confirmed to be uninformative noise (Section 4) |
| `product_rating` | **Cap values above 5 to 5** | Ratings are bound to a 0-5 scale |
| `delivery_delay_days` | **Leave unchanged** | Interpreted as a signed metric: negative = delivered early, positive = delivered late. This is a legitimate business signal, not an error |
| `customer_age` | **Leave unchanged, flag as a limitation** | Ages 10-17 (1.05% of rows) are internally plausible and can't be verified as errors vs. legitimate guardian-linked accounts |
| Categorical fields | **Strip whitespace, lowercase (defensive)** | No issues found, but this guards against future re-runs on messier exports |

Let's apply these one at a time.


In [23]:
df_clean = df.copy()

# --- Categorical standardization (defensive) ---
for c in cat_cols:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.lower()


In [24]:
# --- Floor impossible negative values at 0 ---
floor_cols = ['product_price', 'discount_percent', 'past_return_rate','session_length_minutes', 'num_product_views']

for c in floor_cols:
    n_bad = (df_clean[c] < 0).sum()
    df_clean[c] = df_clean[c].clip(lower=0)
    print(f"{c:25s} floored {n_bad} negative values to 0")

# num_product_views is a count -> must be a whole number
df_clean['num_product_views'] = df_clean['num_product_views'].round().astype(int)


product_price             floored 27285 negative values to 0
discount_percent          floored 1686 negative values to 0
past_return_rate          floored 2220 negative values to 0
session_length_minutes    floored 1487 negative values to 0
num_product_views         floored 21443 negative values to 0


In [26]:
df_clean['session_length_minutes'].min()

np.float64(0.0)

In [27]:
# --- Cap product_rating to the valid 0-5 scale ---
n_high = (df_clean['product_rating'] > 5).sum()
df_clean['product_rating'] = df_clean['product_rating'].clip(lower=0, upper=5)
print(f"product_rating: capped {n_high} values above 5 down to 5")


product_rating: capped 3831 values above 5 down to 5


In [28]:
# --- delivery_delay_days: intentionally left unchanged ---
# Negative = delivered early, positive = delivered late. This is a signed business
# metric, not an error, so no correction is applied here.
n_early = (df_clean['delivery_delay_days'] < 0).sum()
print(f"delivery_delay_days: kept as-is. {n_early} rows reflect early deliveries (negative values).")


delivery_delay_days: kept as-is. 78564 rows reflect early deliveries (negative values).


In [29]:
# --- customer_age: intentionally left unchanged, documented as a limitation ---
n_minor = (df_clean['customer_age'] < 18).sum()
print(f"customer_age: {n_minor} rows ({n_minor/len(df_clean)*100:.2f}%) show age < 18.")
print("Retained without modification — plausible values (min=10), not verifiable as errors.")
print("Documented as a limitation in the Data Quality Report.")


customer_age: 2109 rows (1.05%) show age < 18.
Retained without modification — plausible values (min=10), not verifiable as errors.
Documented as a limitation in the Data Quality Report.


## 7. Final Formatting

Round continuous fields to sensible precision, enforce integer dtypes where the field is a true count, and reorder columns so the primary key leads and the target variable (`returned`) trails — a small readability convention, not a substantive change.


In [30]:
# Round continuous fields
df_clean['product_price'] = df_clean['product_price'].round(2)
df_clean['discount_percent'] = df_clean['discount_percent'].round(2)
df_clean['product_rating'] = df_clean['product_rating'].round(2)
df_clean['past_return_rate'] = df_clean['past_return_rate'].round(4)
df_clean['delivery_delay_days'] = df_clean['delivery_delay_days'].round(2)
df_clean['session_length_minutes'] = df_clean['session_length_minutes'].round(2)

# Enforce integer dtypes for true count/flag fields
int_cols = ['customer_age', 'past_purchase_count', 'num_product_views',
            'used_coupon', 'returned', 'order_id']
for c in int_cols:
    df_clean[c] = df_clean[c].astype(int)

# Reorder: order_id first, returned (target) last
cols = ['order_id'] + [c for c in df_clean.columns if c not in ('order_id', 'returned')] + ['returned']
df_clean = df_clean[cols]

df_clean.head()


,order_id,customer_age,product_price,discount_percent,product_rating,past_purchase_count,past_return_rate,delivery_delay_days,session_length_minutes,num_product_views,device_type,product_category,shipping_method,payment_method,used_coupon,returned
0,75381,25,11.67,43.56,1.93,8,0.3314,-0.09,3.27,26,tablet,sports,express,paypal,1,0
1,65569,76,31.38,66.53,0.69,9,0.1052,3.64,130.51,8,mobile,electronics,standard,credit_card,1,0
2,163473,40,25.46,20.76,2.35,14,0.2367,0.29,92.78,0,desktop,toys,express,credit_card,0,0
3,90518,62,13.52,25.67,2.03,9,0.4100,1.68,103.00,7,mobile,clothing,express,apple_pay,0,0
4,138866,51,75.77,15.53,4.91,9,0.1236,-0.15,100.75,6,mobile,electronics,standard,credit_card,1,0


## 8. Verification

Confirm every impossible value has been corrected and nothing else changed unexpectedly (row count, column count, no new missing values).


In [31]:
print("Shape unchanged:", df_clean.shape == df.shape, df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print()

for c in floor_cols:
    print(f"{c:25s} negatives remaining: {(df_clean[c] < 0).sum()}")
print(f"{'product_rating':25s} values > 5 remaining: {(df_clean['product_rating'] > 5).sum()}")
print()

df_clean.describe().T[['min', 'max']]


Shape unchanged: True (200000, 16)
Missing values: 0

product_price             negatives remaining: 0
discount_percent          negatives remaining: 0
past_return_rate          negatives remaining: 0
session_length_minutes    negatives remaining: 0
num_product_views         negatives remaining: 0
product_rating            values > 5 remaining: 0



,min,max
order_id,0.00,249999.0000
customer_age,10.00,87.0000
product_price,0.00,855.0800
discount_percent,0.00,78.7900
product_rating,0.45,5.0000
past_purchase_count,1.00,26.0000
past_return_rate,0.00,0.8664
delivery_delay_days,-9.05,7.9600
session_length_minutes,0.00,137.6500
num_product_views,0.00,44.0000


**Result:** no negative values remain in any of the five floored fields, no ratings exceed 5, shape is unchanged (200,000 rows × 16 columns), and no missing values were introduced. `delivery_delay_days` and `customer_age` retain their original ranges by design.

## 9. Save the Cleaned Dataset


In [34]:
df_clean.to_csv("/Users/venugopalnaidupotturi/Downloads/cleaned.csv", index=False)
print(f"Final shape: {df_clean.shape}")


Final shape: (200000, 16)


## Summary

| Check | Result |
|---|---|
| Missing values | 0 |
| Duplicate rows / order IDs | 0 |
| Negative values corrected | `product_price` (13.64%), `num_product_views` (10.72%), `past_return_rate` (1.11%), `discount_percent` (0.84%), `session_length_minutes` (0.74%) |
| Out-of-range values corrected | `product_rating` > 5 (1.92%) |
| Fields intentionally left unchanged | `delivery_delay_days` (signed metric), `customer_age` (plausible, unverifiable) |
| Final dataset | 200,000 rows × 16 columns, analysis-ready |

This cleaned file (`ecommerce_returns_cleaned.csv`) is the input for the Exploratory Data Analysis stage that follows.
